MODULES IMPORTATION

In [45]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset

import accelerate
import torch
import transformers
import evaluate
import numpy as np

CLEANING AND LOADING DATA

In [ ]:
# Small transformation of the df
df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
df.rename(columns={"oare_id":"sentence_id", "transliteration":"akkadian", "translation":"english"}, inplace=True)
df.info()

# Splitting the df in a train & test sets 
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Converting into datasets (it's the format used by HF for the training)
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

<class 'pandas.DataFrame'>
RangeIndex: 1561 entries, 0 to 1560
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sentence_id  1561 non-null   str  
 1   akkadian     1561 non-null   str  
 2   english      1561 non-null   str  
dtypes: str(3)
memory usage: 1.6 MB


In [60]:
model_name = "google/byt5-small"   # ou autre petit seq2seq
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.gradient_checkpointing_enable()
model.config.use_cache = False

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [61]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

In [62]:
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

In [66]:
SOURCE_COL = "akkadian"
TARGET_COL = "english"

max_source_length = 128
max_target_length = 128

def preprocess(batch):
    model_inputs = tokenizer(
        batch[SOURCE_COL],
        max_length=max_source_length,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch[TARGET_COL],
        max_length=max_target_length,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [67]:
tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tokenized_val   = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

In [70]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

In [71]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=True,                  # si GPU compatible
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    fp16_full_eval=False,
    optim="adafactor"
)

In [73]:
import evaluate
import numpy as np

bleu = evaluate.load("bleu")
chrf = evaluate.load("chrf")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu_score = bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )["bleu"]

    chrf_score = chrf.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )["score"]

    return {"bleu": bleu_score, "chrf": chrf_score}

In [74]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [75]:
import torch
torch.cuda.empty_cache()

In [76]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss


ZeroDivisionError: division by zero